# Pacotes

In [2]:
#! pip install scikeras

In [3]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from prettytable import PrettyTable

import sklearn
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV

import time
from tensorflow import keras
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, LSTM, Bidirectional, Flatten #Global max pullin avg max pulling
from keras.layers import  Masking
from keras.regularizers import l2
from scikeras.wrappers import KerasClassifier


from keras.callbacks import EarlyStopping, ModelCheckpoint

tf.config.list_physical_devices('GPU')

2023-04-20 12:06:11.833647: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:267] failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2023-04-20 12:06:11.833701: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: pdpachuvas
2023-04-20 12:06:11.833710: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: pdpachuvas
2023-04-20 12:06:11.833824: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported version is: 525.105.17
2023-04-20 12:06:11.833861: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:204] kernel reported version is: 525.89.2
2023-04-20 12:06:11.833867: E tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:313] kernel version 525.89.2 does not match DSO version 525.105.17 -- cannot find working devices in this configuration


[]

# Funcoes

In [4]:
##############################
###### Matriz de confusão ####
##############################

def matriz_confusao(y_real,y_predito,modelo):

### Grafico ###

  tabela=confusion_matrix(y_real,y_predito)

  group_names = ["True Neg","False Pos","False Neg","True Pos"]
  group_counts = ["{0:0.0f}".format(value) for value in
                tabela.flatten()]
  group_percentages = ["{0:.5%}".format(value) for value in
                     tabela.flatten()/np.sum(tabela)]
  labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
          zip(group_names,group_counts,group_percentages)]
  labels = np.asarray(labels).reshape(2,2)
  f = plt.figure()
  f.set_figwidth(8)
  f.set_figheight(8)

  sns.heatmap(tabela, annot=labels, fmt="", cmap='Blues')

### Tabela ###
  Resultados=PrettyTable()
  Resultados.field_names=["Métrica","Resultado"]
  Resultados.title= modelo
  Resultados.align["Métrica"]="l"
  Resultados.align["Resultado"]="r"

  Resultados.add_row(["Acurácia:",round(sklearn.metrics.accuracy_score(y_real,y_predito),2)])
  Resultados.add_row(["Precisão:",round(sklearn.metrics.precision_score(y_real,y_predito),2)])
  Resultados.add_row(["Recall:",round(sklearn.metrics.recall_score(y_real,y_predito),2)])
  Resultados.add_row(["F1-Score:",round(sklearn.metrics.f1_score(y_real,y_predito),2)])

  print(Resultados)
  
  return



    
def transformacao_estrutura(df, variaveis):
    '''
    Extrai os atributos do dataframe pandas

    Params:
    -------
    df: dataframe de entrada
    variaveis = lista de entrada
    Returns:
    par X, Y
    '''

    X = df[variaveis]

    Y = df[['ocorrencia']]
    
    n_tempo, n_coord = conta_tempos_e_coordenadas(df)
    
    X_transformado = X.to_numpy().astype(np.float32).reshape(n_tempo , n_coord, -1)
    Y_transformado = Y.to_numpy().astype(np.float32).reshape(n_tempo , n_coord, -1)
    
    return X_transformado, Y_transformado

    
def conta_tempos_e_coordenadas(df):
    '''
    Conta o número de tempos e coordenadas de um dataframe

    Params:
    -------
    df: dataframe com os atributos 'tempo', e 'n_coord'

    Returns:
    --------
    tupla número de tempos, número de coordenadas
    '''
    tempo = len(df['tempo'].unique())
    n_coord = len(df['n_coord'].unique())

    return tempo, n_coord


def calcula_sample_weight(df,peso):
    '''
    Calcula os pesos para as amostras de treinamento
    '''
    
    Y = df[['ocorrencia']]
    sample_weight_ = list(Y['ocorrencia'])
    
    series = pd.Series(sample_weight_)
    series = series*peso #3191848/613
    series = series+1
    series = series/(peso+1)
    
    sample_weight_ = np.array(list(series)).astype(np.float32)
    
    return sample_weight_
    

# Importacao dos dados

In [5]:
df = pd.read_csv('df_comp.csv')

In [6]:
df.replace(np.nan,-1, inplace=True)

In [7]:
df['tempo'] =  pd.to_datetime(df['tempo'])

In [8]:
df = df.sort_values(['tempo','n_coord'])

In [39]:
variaveis = [
        "vintequatrohoras"
            , "Altitude_numerica"
             , "Declividade_numerica"
             , "graurisc"
             , 'lon_ocr'
             , 'lat_ocr'
             , 'Orientacao_numerica'
             , 'Curv_Vertical_numerica'
             , 'Curv_Horizontal_numerica'
             , 'Relevo_sombreado_numerico'
             ,'Vegetacao_Natural_Dominante'                
     ,'Area_Antropica_Dominante'                   
     ,'legenda_2_Pecuária (pastagens)'             
     ,'Floresta_Ombrofila_Densa'                   
     ,'Formacao_Pioneira'                          
     ,'Floresta_Ombrofila_Densa_Submontana'        
     ,'Influencia_urbana'                          
     ,'Vegetacao_Secundaria'                       
     ,'Argilossolo'                                
     ,'Gleissolo'                                  
     ,'Argilossolo_Vermelho_Amarelo'               
     ,'Gleissolo_Melanico'                         
     ,'Area_Urbana'                                
     ,'Unidades_de_Conservacao_Protecao_Integral'  
     ,'Plano_de_Manejo'                            
     ,'flg_comunidades'                          
     ,'flg_agricola'                               
     ,'flg_exploracao_mineral'                     
     ,'flg_rocha'                                  
     ,'flg_cobertura_vegetal'                      
     ,'flg_afloramento_rochoso'                    
     ,'flg_favela'                                 
     ,'flg_ocupacao_desordenada' 
           ]

In [9]:
teste = df[df.loc[:,'tempo'] > pd.to_datetime('2021-08-24 09:00:00')]

In [10]:
treino = df[df.loc[:,'tempo'] <= pd.to_datetime('2021-02-24 09:00:00')]

In [11]:
filtro_inferior = pd.to_datetime('2021-02-24 09:00:00') < df.loc[:, 'tempo']
filtro_superior = df.loc[:, 'tempo'] <= pd.to_datetime('2021-08-24 09:00:00')
validacao = df[filtro_inferior & filtro_superior]

In [12]:
### Treino

X_train, Y_train = transformacao_estrutura(treino)

### Validação

X_val, Y_val = transformacao_estrutura(validacao)

### Teste

X_teste, Y_teste = transformacao_estrutura(teste)

class_weight_ = {0: (1/3191848),
                1: (1/613)}

In [13]:
qtd_o = sum(treino['ocorrencia'])
tam_o = len(treino['ocorrencia'])
print(qtd_o)
print(tam_o)
print(tam_o/qtd_o) #peso

284
1555500
5477.112676056338


In [18]:
sample_weight_ = np.asarray(calcula_sample_weight(treino,5590)).astype(np.float32)

In [19]:
print(sample_weight_.max())
print(sample_weight_.min())

1.0
0.00017885889


In [20]:
sample_weight_transformado = sample_weight_.reshape(732, 2125, -1)

In [21]:
print(sample_weight_transformado.shape)
print(sample_weight_transformado.shape)
print(sample_weight_transformado.shape)

(732, 2125, 1)
(732, 2125, 1)
(732, 2125, 1)


In [28]:
# create Weights
    
sample_weight_ = np.asarray(calcula_sample_weight(treino,5600)).astype(np.float32)
    
# create model
    
time_step = X_train.shape[1] # Quantidade de coordenada para equivaler a 1 espaço de tempo
input_dim = X_train.shape[2] #qtd colunas (features)
out = Y_train.shape[2]

In [29]:
print(time_step, input_dim,  out)

2125 10 1


In [50]:
# fix random seed for reproducibility
seed = 42
tf.random.set_seed(seed)

In [51]:
# Function to create model, required for KerasClassifier

def create_model(neurons):
    
    
    ### Treino

    X_train, Y_train = transformacao_estrutura(treino)

    ### Validação

    X_val, Y_val = transformacao_estrutura(validacao)

    ### Teste

    X_teste, Y_teste = transformacao_estrutura(teste)
    
    time_step = X_train.shape[1] # Quantidade de coordenada para equivaler a 1 espaço de tempo
    input_dim = X_train.shape[2] #qtd colunas (features)
    out = Y_train.shape[2]

    # LSTM
    start = time.time()
    model = Sequential()
    model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim,))) #camada de entrada
    model.add(LSTM(neurons,activation='elu'
                   , input_shape=(time_step, input_dim,)
                   ,return_sequences=True
                   , go_backwards=True
                   , kernel_regularizer=l2(0.01)
                   , recurrent_regularizer=l2(0.01)
                   , bias_regularizer=l2(0.01))) # camada escondida
    model.add(Dense(out, activation='sigmoid')) #camada saida

    opt = tf.keras.optimizers.Adam()

    model.compile(loss = tf.keras.losses.BinaryCrossentropy(from_logits=False) #https://keras.io/api/losses/probabilistic_losses/ 
                  , optimizer= opt #'adam'
                 # , weighted_metrics=[tf.keras.losses.BinaryCrossentropy(from_logits=False),'accuracy'
                 # , tf.keras.metrics.Precision()
                 # , tf.keras.metrics.Recall()]
                  , sample_weight_mode="temporal")   #Weights

    return model

In [52]:
# define the grid search parameters

dic_parametros = {
    'model__neurons': [32, 37, 42, 47, 52, 57, 64],
    'optimizer__learning_rate': [0.001, 0.01, 0.1, 0.2, 0.3],
    'batch_size': [10, 20, 40, 60, 80, 100],
    'epochs': [10, 50, 100]
    }
    



In [53]:
xtreino = treino[[
        "vintequatrohoras"
            , "Altitude_numerica"
             , "Declividade_numerica"
             , "graurisc"
             , 'lon_ocr'
             , 'lat_ocr'
             , 'Orientacao_numerica'
             , 'Curv_Vertical_numerica'
             , 'Curv_Horizontal_numerica'
             , 'Relevo_sombreado_numerico'
             ,'Vegetacao_Natural_Dominante'                
     ,'Area_Antropica_Dominante'                   
     ,'legenda_2_Pecuária (pastagens)'             
     ,'Floresta_Ombrofila_Densa'                   
     ,'Formacao_Pioneira'                          
     ,'Floresta_Ombrofila_Densa_Submontana'        
     ,'Influencia_urbana'                          
     ,'Vegetacao_Secundaria'                       
     ,'Argilossolo'                                
     ,'Gleissolo'                                  
     ,'Argilossolo_Vermelho_Amarelo'               
     ,'Gleissolo_Melanico'                         
     ,'Area_Urbana'                                
     ,'Unidades_de_Conservacao_Protecao_Integral'  
     ,'Plano_de_Manejo'                            
     ,'flg_comunidades'                          
     ,'flg_agricola'                               
     ,'flg_exploracao_mineral'                     
     ,'flg_rocha'                                  
     ,'flg_cobertura_vegetal'                      
     ,'flg_afloramento_rochoso'                    
     ,'flg_favela'                                 
     ,'flg_ocupacao_desordenada' 
           ]]
ytreino = treino[[
        'ocorrencia' 
           ]]
xvalida = validacao[[
        "vintequatrohoras"
            , "Altitude_numerica"
             , "Declividade_numerica"
             , "graurisc"
             , 'lon_ocr'
             , 'lat_ocr'
             , 'Orientacao_numerica'
             , 'Curv_Vertical_numerica'
             , 'Curv_Horizontal_numerica'
             , 'Relevo_sombreado_numerico'
             ,'Vegetacao_Natural_Dominante'                
     ,'Area_Antropica_Dominante'                   
     ,'legenda_2_Pecuária (pastagens)'             
     ,'Floresta_Ombrofila_Densa'                   
     ,'Formacao_Pioneira'                          
     ,'Floresta_Ombrofila_Densa_Submontana'        
     ,'Influencia_urbana'                          
     ,'Vegetacao_Secundaria'                       
     ,'Argilossolo'                                
     ,'Gleissolo'                                  
     ,'Argilossolo_Vermelho_Amarelo'               
     ,'Gleissolo_Melanico'                         
     ,'Area_Urbana'                                
     ,'Unidades_de_Conservacao_Protecao_Integral'  
     ,'Plano_de_Manejo'                            
     ,'flg_comunidades'                          
     ,'flg_agricola'                               
     ,'flg_exploracao_mineral'                     
     ,'flg_rocha'                                  
     ,'flg_cobertura_vegetal'                      
     ,'flg_afloramento_rochoso'                    
     ,'flg_favela'                                 
     ,'flg_ocupacao_desordenada' 
           ]]
yvalida = validacao[[
        'ocorrencia' 
           ]]

In [54]:
# create model
model = KerasClassifier(model=create_model, verbose=0)


# Grid search parameters

grid = GridSearchCV(estimator=model, param_grid=dic_parametros, n_jobs=-1, cv=10, error_score='raise')
grid_result = grid.fit(xtreino
                 , ytreino
#                  , validation_split=.2
                 , verbose=1     
                 ,validation_data=(xvalida, yvalida)
                 , sample_weight=sample_weight_transformado
                )     




2023-04-20 12:45:26.587418: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-04-20 12:45:26.597920: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-04-20 12:45:26.618547: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, reb

2023-04-20 12:45:26.794544: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-04-20 12:45:26.798525: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-04-20 12:45:26.799028: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-04-20 12:45:26.799155: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudar

2023-04-20 12:45:27.093642: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-04-20 12:45:27.099160: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-04-20 12:45:27.099384: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-04-20 12:45:27.103849: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`

2023-04-20 12:45:27.926093: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-04-20 12:45:27.926393: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory
2023-04-20 12:45:27.926405: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make sure the missing libraries mentioned above are installed properly.
2023-04-20 12:45:27.944402: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory


2023-04-20 12:45:37.432769: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:267] failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2023-04-20 12:45:37.433503: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: pdpachuvas
2023-04-20 12:45:37.433528: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: pdpachuvas
2023-04-20 12:45:37.434149: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported version is: 525.105.17
2023-04-20 12:45:37.434278: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:204] kernel reported version is: 525.89.2
2023-04-20 12:45:37.434331: E tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:313] kernel version 525.89.2 does not match DSO version 525.105.17 -- cannot find working devices in this configuration
2023-04-20

2023-04-20 12:45:37.900527: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:267] failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2023-04-20 12:45:37.900516: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:267] failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2023-04-20 12:45:37.900627: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: pdpachuvas
2023-04-20 12:45:37.900636: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: pdpachuvas
2023-04-20 12:45:37.901031: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported version is: 525.105.17
2023-04-20 12:45:37.901107: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:204] kernel reported version is: 525.89.2
2023-04-

2023-04-20 12:45:38.121086: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:267] failed call to cuInit: CUDA_ERROR_COMPAT_NOT_SUPPORTED_ON_DEVICE: forward compatibility was attempted on non supported HW
2023-04-20 12:45:38.122517: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: pdpachuvas
2023-04-20 12:45:38.122647: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: pdpachuvas
2023-04-20 12:45:38.123212: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported version is: 525.105.17
2023-04-20 12:45:38.123420: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:204] kernel reported version is: 525.89.2
2023-04-20 12:45:38.123479: E tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:313] kernel version 525.89.2 does not match DSO version 525.105.17 -- cannot find working devices in this configuration
2023-04-20

ValueError: Found array with dim 3. None expected <= 2.

In [ ]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))


In [ ]:
time_step = X_train.shape[1] # Quantidade de coordenada para equivaler a 1 espaço de tempo
input_dim = X_train.shape[2] #qtd colunas (features)
out = Y_train.shape[2]

# LSTM
start = time.time()
model = Sequential()
model.add(Masking(mask_value=-1.,input_shape=(time_step, input_dim,))) #camada de entrada
model.add(LSTM(32,activation='elu', input_shape=(time_step, input_dim,),return_sequences=True, go_backwards=True, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01), bias_regularizer=l2(0.01))) # camada escondida
model.add(Dense(out, activation='sigmoid')) #camada saida

opt = tf.keras.optimizers.Adam(learning_rate=0.01)

model.compile(loss = tf.keras.losses.BinaryCrossentropy(from_logits=False) #https://keras.io/api/losses/probabilistic_losses/ 
              , optimizer= opt #'adam'
              , weighted_metrics=[tf.keras.losses.BinaryCrossentropy(from_logits=False),'accuracy'
              , tf.keras.metrics.Precision()
              , tf.keras.metrics.Recall()]
              , sample_weight_mode="temporal")   #Weights
model.summary()
hist = model.fit(treino[]
                 , Y_train
                 , epochs=100
#                  , validation_split=.2
                 ,validation_data=(X_val, Y_val)
                 , verbose=1, batch_size=64
                 , sample_weight=sample_weight_transformado 
                 #(samples, sequence_length)
        )
end = time.time()
print("Total compile time: --------", end - start, 's')



In [ ]:
# summarize history for loss
plt.plot(hist.history['loss'])
plt.plot(hist.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# summarize history for binary crossentropy
plt.plot(hist.history['binary_crossentropy'])
plt.plot(hist.history['val_binary_crossentropy'])
plt.title('model binary crossentropy')
plt.ylabel('binary crossentropy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()


# summarize history for accuracy
plt.plot(hist.history['accuracy'])
plt.plot(hist.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# summarize history for precision
plt.plot(hist.history['precision_6'])
plt.plot(hist.history['val_precision_6'])
plt.title('model precision')
plt.ylabel('precision')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

# summarize history for recall
plt.plot(hist.history['recall_6'])
plt.plot(hist.history['val_recall_6'])
plt.title('model recall')
plt.ylabel('recall')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
model.summary()

In [ ]:
predictions = (model.predict(X_teste, verbose=1) > 0.5).astype("int32").reshape((1,-1))

In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]

In [ ]:
pred_y = pd.DataFrame(flatten(predictions), columns=['Prediction']) 

In [ ]:
matriz_confusao(teste[['ocorrencia']],pred_y,'1')